# Create TWAS Awards (PRIZE PATTERN)

TWAS Award / Prize recipients from The World Academy of Sciences official recipient archive and official TWAS announcement pages.

**Prerequisite:** run `scripts/local/twas_awards_to_s3.py` to fetch the official TWAS pages, validate row uniqueness, write parquet, and upload to S3. Contractor-local validation used `--skip-upload`; admin must run the upload because contractor access has no AWS/S3 credentials.

**Sources:**
- `https://twas.org/recipients-twas-awards-and-prizes`
- linked official TWAS announcement pages for 2010-2022
- `https://twas.org/article/winners-2024-twas-awards-announced`
- `https://twas.org/article/twas-announces-2024-slate-awards`

**S3 location:** `s3://openalex-ingest/awards/twas_awards/twas_awards.parquet`

**OpenAlex funder:** The World Academy of Sciences, `funder_id = 4320321078`, ROR `https://ror.org/00twbd828`, DOI `10.13039/501100002222`.

**Priority:** 72. Priorities 66-71 are reserved for in-flight BBVA, Millennium, Dan David, Crafoord, Kyoto, and Royal Society PRs.

**Amount policy:** prize-pattern waiver applies. Older archive rows and 2010-2011 pages do not publish reliable amounts, so `amount` remains NULL. Official announcement pages from 2012 onward publish USD cash amounts for many rows; the script stores `amount_per_laureate` after applying the documented shared-prize rule.

## Step 1: Create Raw Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.twas_awards_raw
USING delta AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/twas_awards/twas_awards.parquet`;


In [ ]:
%sql
SELECT COUNT(*) AS raw_rows
FROM openalex.awards.twas_awards_raw;


## Step 1.5: Inspect Raw Data First

Verify exact source columns and values before relying on the transform below. The source script writes all raw columns as strings with `df.astype("string")` before parquet.

In [ ]:
%sql
DESCRIBE openalex.awards.twas_awards_raw;


In [ ]:
%sql
SELECT *
FROM openalex.awards.twas_awards_raw
LIMIT 10;


In [ ]:
%sql
-- Required money-field scan. TWAS is a prize source: NULL amounts are allowed
-- when the official page does not publish a reliable per-laureate value.
SELECT column_name
FROM (DESCRIBE openalex.awards.twas_awards_raw)
WHERE LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|fund_|funding|cost|budget|grant_offer|awarded|money|cash|currency|usd';


In [ ]:
%sql
SELECT
    award_year,
    COUNT(*) AS rows,
    COUNT(amount_per_laureate) AS rows_with_amount,
    collect_set(currency) AS currencies,
    MIN(TRY_CAST(amount_per_laureate AS DOUBLE)) AS min_amount,
    MAX(TRY_CAST(amount_per_laureate AS DOUBLE)) AS max_amount,
    AVG(TRY_CAST(amount_per_laureate AS DOUBLE)) AS avg_amount
FROM openalex.awards.twas_awards_raw
GROUP BY award_year
ORDER BY TRY_CAST(award_year AS INT);


In [ ]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(funder_award_id) AS has_funder_award_id,
    COUNT(award_year) AS has_year,
    COUNT(award_name) AS has_award_name,
    COUNT(award_field) AS has_award_field,
    COUNT(laureate_name) AS has_laureate_name,
    COUNT(citation) AS has_citation,
    COUNT(description) AS has_description,
    COUNT(amount_per_laureate) AS has_amount,
    COUNT(currency) AS has_currency,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    MIN(TRY_CAST(award_year AS INT)) AS min_year,
    MAX(TRY_CAST(award_year AS INT)) AS max_year
FROM openalex.awards.twas_awards_raw;


In [ ]:
%sql
SELECT
    source_page_title,
    parser_path,
    COUNT(*) AS rows,
    MIN(TRY_CAST(award_year AS INT)) AS min_year,
    MAX(TRY_CAST(award_year AS INT)) AS max_year
FROM openalex.awards.twas_awards_raw
GROUP BY source_page_title, parser_path
ORDER BY min_year, source_page_title, parser_path;


## Step 1.6: Funder Existence Fail-Fast

Must return exactly one row. If zero rows, stop before transforming because the CROSS JOIN would emit zero awards.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320321078;


## Step 2: Transform to OpenAlex Awards Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.twas_awards
USING delta
AS
WITH twas_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320321078
),
normalized AS (
    SELECT
        r.*,
        TRY_CAST(r.award_year AS INT) AS award_year_int,
        TRY_CAST(r.amount_per_laureate AS DOUBLE) AS amount_double,
        NULLIF(TRIM(r.currency), '') AS currency_norm,
        NULLIF(TRIM(r.award_name), '') AS award_name_norm,
        NULLIF(TRIM(r.award_field), '') AS award_field_norm,
        NULLIF(TRIM(r.citation), '') AS citation_norm,
        NULLIF(TRIM(r.description), '') AS description_norm,
        NULLIF(TRIM(r.laureate_name), '') AS laureate_name_norm,
        NULLIF(TRIM(r.laureate_given_name), '') AS given_name_norm,
        NULLIF(TRIM(r.laureate_family_name), '') AS family_name_norm,
        NULLIF(TRIM(r.laureate_affiliation_or_context), '') AS affiliation_norm,
        NULLIF(TRIM(r.laureate_country_or_nationality), '') AS country_norm,
        NULLIF(TRIM(r.landing_page_url), '') AS landing_page_url_norm
    FROM openalex.awards.twas_awards_raw r
),
awards AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':twas-awards:', LOWER(n.funder_award_id)))) % 9000000000 AS id,
        CONCAT(CAST(n.award_year_int AS STRING), ' ', n.award_name_norm, ' - ', n.laureate_name_norm) AS display_name,
        COALESCE(n.citation_norm, n.description_norm) AS description,
        f.funder_id,
        n.funder_award_id,
        n.amount_double AS amount,
        n.currency_norm AS currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) AS id,
            f.display_name,
            f.ror_id,
            f.doi
        ) AS funder,
        'prize' AS funding_type,
        n.award_name_norm AS funder_scheme,
        'twas_awards' AS provenance,
        TRY_TO_DATE(CONCAT(CAST(n.award_year_int AS STRING), '-01-01'), 'yyyy-MM-dd') AS start_date,
        TRY_TO_DATE(CONCAT(CAST(n.award_year_int AS STRING), '-12-31'), 'yyyy-MM-dd') AS end_date,
        n.award_year_int AS start_year,
        n.award_year_int AS end_year,
        struct(
            n.given_name_norm AS given_name,
            n.family_name_norm AS family_name,
            CAST(NULL AS STRING) AS orcid,
            CAST(NULL AS DATE) AS role_start,
            struct(
                n.affiliation_norm AS name,
                n.country_norm AS country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        ) AS lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) AS co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) AS investigators,
        n.landing_page_url_norm AS landing_page_url,
        CAST(NULL AS STRING) AS doi,
        current_timestamp() AS created_date,
        current_timestamp() AS updated_date
    FROM normalized n
    CROSS JOIN twas_funder f
    WHERE n.funder_award_id IS NOT NULL
      AND n.award_year_int IS NOT NULL
      AND n.award_name_norm IS NOT NULL
      AND n.award_field_norm IS NOT NULL
      AND n.laureate_name_norm IS NOT NULL
)
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G', CAST(id AS STRING)) AS works_api_url,
    created_date,
    updated_date
FROM awards;


In [ ]:
%sql
SELECT
    COUNT(*) AS award_rows,
    COUNT(DISTINCT id) AS distinct_ids,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    MIN(start_year) AS min_year,
    MAX(start_year) AS max_year,
    COUNT(amount) AS rows_with_amount,
    COUNT(currency) AS rows_with_currency,
    COUNT(description) AS rows_with_description,
    COUNT(lead_investigator.family_name) AS rows_with_family_name
FROM openalex.awards.twas_awards;


In [ ]:
%sql
SELECT id, COUNT(*) AS rows
FROM openalex.awards.twas_awards
GROUP BY id
HAVING COUNT(*) > 1;


## Step 3: Delete and Insert into `openalex_awards_raw`

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'twas_awards' AND priority = 72;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    72 AS priority
FROM openalex.awards.twas_awards;


## Step 4: Verification

In [ ]:
%sql
SELECT provenance, priority, COUNT(*) AS rows
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'twas_awards' AND priority = 72
GROUP BY provenance, priority;


In [ ]:
%sql
SELECT
    start_year,
    COUNT(*) AS rows,
    COUNT(amount) AS rows_with_amount,
    collect_set(currency) AS currencies
FROM openalex.awards.twas_awards
GROUP BY start_year
ORDER BY start_year;


In [ ]:
%sql
SELECT
    funder_scheme,
    COUNT(*) AS rows,
    MIN(start_year) AS min_year,
    MAX(start_year) AS max_year
FROM openalex.awards.twas_awards
GROUP BY funder_scheme
ORDER BY rows DESC, funder_scheme;


In [ ]:
%sql
SELECT *
FROM openalex.awards.twas_awards
ORDER BY start_year DESC, funder_scheme, display_name
LIMIT 50;


## Admin Handoff Notes

Contractor-local validation completed with `scripts/local/twas_awards_to_s3.py --skip-upload --no-cache`: 292 rows, 292 distinct `funder_award_id` values, year range 1985-2026, and 129 rows with official USD amount coverage. Admin still needs to upload the parquet to S3, run this notebook in Databricks, inspect the verification cells above, and only then add the source to the scheduled award job if QA passes.